# 01 · Data pipeline

Downloads LEAD 1.0 and BDG2 metadata/weather, builds the processed tables, checks the
building split, writes the data quality report and, optionally, loads everything into the
shared Postgres database.

* Runtime: CPU is enough, about 5 minutes on a fresh Colab runtime.
* Colab secrets (key icon on the left): `GITHUB_TOKEN` while the repository is private,
  `DATABASE_URL` only for the optional database step.
* Every step calls the same code as the command line (`python -m src.data.<step>`).

## 0. Setup

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO, BRANCH = "Ozgurisikdamar/enerji-anomali", "feature/data-ml"
USE_DRIVE = False  # keep data/ and models_out/ on Google Drive between Colab runtimes
IN_COLAB = "google.colab" in sys.modules


def colab_secret(name):
    from google.colab import userdata

    try:
        return userdata.get(name)
    except Exception:  # secret missing or notebook access not granted
        return None


if IN_COLAB:
    root = Path("/content/enerji-anomali")
    if not root.exists():
        url = f"https://github.com/{REPO}.git"
        token = colab_secret("GITHUB_TOKEN")
        auth_url = url.replace("https://", f"https://x-access-token:{token}@") if token else url
        clone = subprocess.run(
            ["git", "clone", "-q", "-b", BRANCH, auth_url, str(root)],
            capture_output=True,
            text=True,
        )
        if clone.returncode:
            raise RuntimeError("clone failed: check the GITHUB_TOKEN secret and repository access")
        subprocess.run(["git", "-C", str(root), "remote", "set-url", "origin", url], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(root / "requirements-colab.txt")],
        check=True,
    )
    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        store = Path("/content/drive/MyDrive/enerji-anomali")
        os.environ.update(DATA_DIR=str(store / "data"), MODEL_DIR=str(store / "models_out"))
    if url := colab_secret("DATABASE_URL"):
        os.environ["DATABASE_URL"] = url
else:
    root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(root)
sys.path.insert(0, str(root))
print("repo:", root)

In [ ]:
import logging

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

from src.config import get_settings

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
settings = get_settings()
settings.data_dir, settings.reports_dir

## 1. Download raw data

Files are cached in `data/raw` and verified against pinned SHA-256 checksums.

In [ ]:
from src.data import download
from src.data.sources import ALL_SOURCES

paths = download.fetch_all(settings.raw_dir)
pd.DataFrame(
    [
        {
            "source": s.name,
            "file": paths[s.name].name,
            "MB": round(paths[s.name].stat().st_size / 1e6, 1),
        }
        for s in ALL_SOURCES
    ]
)

## 2. Prepare tables

Merges LEAD readings with BDG2 building metadata and weather, converts site 0 from kBTU to
kWh and writes `buildings`, `weather` and `readings` parquet files to `data/processed`.

In [ ]:
from src.data.prepare import prepare

tables = prepare(settings.raw_dir)
tables.save(settings.processed_dir)
for name in tables.FILES:
    print(f"{name:10s} {len(getattr(tables, name)):>10,} rows")
tables.buildings.head()

In [ ]:
tables.readings.head()

## 3. Building split

The split is committed in `data/splits.json` and is by building, never by row, so test
buildings stay unseen. New customer buildings get a stable hash-based split.

In [ ]:
from src.data.split import load_split

split = load_split()
split_of = tables.buildings.assign(split=tables.buildings["building_id"].map(split.as_series()))
split_of.groupby("split").agg(
    buildings=("building_id", "size"), primary_uses=("primary_use", "nunique")
)

## 4. Model frame and data quality report

`model_frame` puts every building on a complete hourly grid and joins metadata, weather and
split. This is the exact frame the ML notebook trains on.

In [ ]:
from src.common.report import write_json
from src.data import quality
from src.data.dataset import model_frame

frame = model_frame(tables)
summary = quality.summarize(frame, tables.buildings)
out = settings.reports_dir / "data"
write_json(out / "quality.json", summary)
(out / "quality.md").write_text(quality.to_markdown(summary))
figures = quality.plot(frame, summary, out / "figures")
display(Markdown(quality.to_markdown(summary)))

In [ ]:
for path in figures:
    display(Image(str(path)))

## 5. Seasonality and weather

Monthly load against outdoor temperature, averaged over all buildings. Heating and cooling
make consumption seasonal, which is why the models use weather and calendar features.

In [ ]:
monthly = (
    frame.assign(month=frame["ts"].dt.month)
    .groupby("month")
    .agg(kwh=("kwh", "mean"), air_temp=("air_temp", "mean"))
)
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(monthly.index, monthly["kwh"], marker="o", label="mean kWh per hour")
ax.set_xlabel("month")
ax.set_ylabel("kWh")
ax2 = ax.twinx()
ax2.plot(monthly.index, monthly["air_temp"], color="tab:red", marker="s", label="air temperature")
ax2.set_ylabel("°C")
fig.legend(loc="upper center", ncol=2)
fig.tight_layout()

## 6. Synthetic waste scenarios

LEAD labels are mostly meter faults. To test waste detection, the ML notebook injects known
patterns into clean windows. Here is one night base load injection.

In [ ]:
from src.data import synthetic

building = split.test[0]
injection = synthetic.plan(frame, "night_base_load", [building], seed=42)[0]
injected = synthetic.inject(frame, [injection])
week = injected[
    (injected["building_id"] == building) & injected["ts"].between(injection.start, injection.end)
]
original = frame.loc[week.index, "kwh"]
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(week["ts"], original, label="original")
ax.plot(week["ts"], week["kwh"], label="with injected night waste", alpha=0.8)
ax.set_title(f"building {building}: {synthetic.SCENARIOS['night_base_load'].description}")
ax.legend()
fig.tight_layout()

## 7. Optional: load into the shared database

Runs only when the `DATABASE_URL` secret points at the database owner. Team roles are created
from `DB_TEAM_USER`/`DB_TEAM_PASSWORD` and `DB_READER_USER`/`DB_READER_PASSWORD` when set.
Loading is idempotent; pass `force=True` to reload.

In [ ]:
from src.common.db import connect
from src.data.load_db import apply_schema, ensure_roles_from_env, load_tables

if url := os.getenv("DATABASE_URL"):
    with connect(url) as conn:
        apply_schema(conn)
        roles = ensure_roles_from_env(conn)
        loaded = load_tables(conn, tables, force=False)
    print("loaded" if loaded else "already loaded", "| roles:", roles or "none")
else:
    print("DATABASE_URL not set, skipping")

## 8. Keep the results

`reports/data` is committed to the repository. In Colab, download it or copy it to Drive.

In [ ]:
if IN_COLAB:
    import shutil

    from google.colab import files

    files.download(
        shutil.make_archive("/content/data_report", "zip", settings.reports_dir / "data")
    )